# 01 - Data exploration
**Project:** E-commerce Sales Dashboard (Online Retail dataset)

**Goal of this notebook:** load the raw data, look at it, and find what is wrong with it. **We do not change anything here.** The fixes are made in `02_cleaning.ipynb`.

## 1. Load the data
The original file is an Excel file, which is slow to read (1 to 2 minutes). The first time, we read it and save a CSV copy. After that, the notebook reads the fast CSV.

> Change the file names below if yours are different. Paths start with `../` because this notebook is inside the `notebooks/` folder.

In [1]:
import os
import pandas as pd

XLSX_PATH = "../data/rawData.xlsx"   # original Excel file
CSV_PATH = "../data/rawData1.csv"    # fast copy

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
else:
    df = pd.read_excel(XLSX_PATH)
    df.to_csv(CSV_PATH, index=False)

print("Loaded from:", CSV_PATH if os.path.exists(CSV_PATH) else XLSX_PATH)

Loaded from: ../data/rawData1.csv


## 2. First look
`head()` shows the first 5 rows. `shape` gives `(rows, columns)`. It has no parentheses because it is a stored value, not an action.

In [2]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
print(df.shape)

(541909, 8)


## 3. Column types and missing values
- `info()` lists each column, its type, and how many values are **not** missing.
- `isna().sum()` counts the missing values per column.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


In [5]:
print(df.isna().sum())

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


## 4. Basic statistics
`describe()` gives the count, mean, std (spread), min, max, and quartiles of the numeric columns. Look at the **min** row for impossible values. `CustomerID` is a label, so ignore its statistics.

In [6]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


## 5. Looking for problems
### 5.1 Negative quantities
A negative quantity means items were returned or adjusted.

In [7]:
negative_qty = df[df["Quantity"] < 0]
print("Rows with negative quantity:", len(negative_qty))

Rows with negative quantity: 10624


### 5.2 Cancelled invoices
According to the dataset documentation, an invoice number starting with `C` is a cancellation. We **check** this rule on our own data instead of trusting it.

In [8]:
cancelled = df[df["InvoiceNo"].str.startswith("C")]
print("Cancelled invoices (start with C):", len(cancelled))
print("Of these, with negative quantity:", (cancelled["Quantity"] < 0).sum())
cancelled.head()

Cancelled invoices (start with C): 9288
Of these, with negative quantity: 9288


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


### 5.3 Negative quantities that are NOT cancellations
`~` means "not", `&` means "and". Each condition needs its own parentheses.

In [9]:
other_negative = df[(df["Quantity"] < 0) & (~df["InvoiceNo"].str.startswith("C"))]
print("Negative quantity, not cancelled:", len(other_negative))
other_negative.head(10)

Negative quantity, not cancelled: 1336


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7192,537000,21414,NaN,-22,2010-12-03 15:32:00,0.0,NaN,United Kingdom
7193,537001,21653,NaN,-6,2010-12-03 15:33:00,0.0,NaN,United Kingdom
7195,537003,85126,NaN,-2,2010-12-03 15:33:00,0.0,NaN,United Kingdom
7196,537004,21814,NaN,-30,2010-12-03 15:34:00,0.0,NaN,United Kingdom
7197,537005,21692,NaN,-70,2010-12-03 15:35:00,0.0,NaN,United Kingdom


### 5.4 Invoice numbers that contain letters
We count how many invoices start with each letter.

In [10]:
letters = df[df["InvoiceNo"].str.contains("[A-Za-z]")]
print(letters["InvoiceNo"].str[0].value_counts())

InvoiceNo
C    9288
A       3
Name: count, dtype: int64


### 5.5 Zero or negative prices

In [11]:
bad_price = df[df["UnitPrice"] <= 0]
print("Rows with price <= 0:", len(bad_price))
bad_price.head()

Rows with price <= 0: 2517


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom


### 5.6 Duplicate rows
`duplicated()` marks a row True if an identical row appeared before it.

In [12]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 5268


### 5.7 Date range
`InvoiceDate` is stored as text. We convert a **copy** to real dates just to look at the range.

In [13]:
dates = pd.to_datetime(df["InvoiceDate"])
print("First date:", dates.min())
print("Last date: ", dates.max())

First date: 2010-12-01 08:26:00
Last date:  2011-12-09 12:50:00


### 5.8 Countries

In [14]:
print(df["Country"].value_counts())

Country
United Kingdom          495478
Germany                   9495
France                    8557
EIRE                      8196
Spain                     2533
Netherlands               2371
Belgium                   2069
Switzerland               2002
Portugal                  1519
Australia                 1259
Norway                    1086
Italy                      803
Channel Islands            758
Finland                    695
Cyprus                     622
Sweden                     462
Unspecified                446
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
Israel                     297
USA                        291
Hong Kong                  288
Singapore                  229
Iceland                    182
Canada                     151
Greece                     146
Malta                      127
United Arab Emirates        68
European Community          61
RSA                         58


### 5.9 Product codes that are not products
Real products have a stock code that starts with 5 digits (for example `85123A`). Codes like `POST` (postage) or `DOT` (fees) are not products.

In [15]:
non_products = df[~df["StockCode"].str.match(r"^\d{5}")]
print(non_products["StockCode"].value_counts().head(15))

StockCode
POST            1256
DOT              710
M                571
C2               144
D                 77
S                 63
BANK CHARGES      37
AMAZONFEE         34
CRUK              16
DCGSSGIRL         13
DCGSSBOY          11
gift_0001_20      10
gift_0001_10       9
gift_0001_30       8
DCGS0003           5
Name: count, dtype: int64


## 6. Summary of the findings
Fill in the numbers from your own outputs.

| Problem | Rows | Decision (applied in `02_cleaning.ipynb`) |
|---|---|---|
| Rows in the raw data | 541,909 | - |
| Missing `Description` | 1,454 | Removed by the price filter, no separate step |
| Missing `CustomerID` | 135,080 (about 25%) | **Remove** (cahier des charges). Every order in the dashboard must belong to a customer. Revenue then covers identified customers only |
| Cancelled invoices (start with `C`) | 9,288 | Remove |
| Negative quantity, not cancelled | 1,336 | Remove |
| Price <= 0 | ... | Remove |
| Duplicate rows | ... | Remove |
| Very large orders cancelled in full (74,215 and 80,995 units) | 2 | Remove the original sale (outliers) |
| Non-product codes (`POST`, `DOT`...) | ... | Keep in the data, exclude only from the product ranking |
| `InvoiceDate` stored as text | all rows | Convert to a real date |
| Date range | ... to ... | December 2011 is incomplete (data stops on 9 Dec) |

**Next step:** `02_cleaning.ipynb`.